In [1]:
import openai, json
import requests

client = openai.OpenAI()
messages = []


In [2]:


def get_popular_movies(): #- /movies에서 인기 영화를 가져옵니다.
    response = requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies")
    return response.json()

def get_movie_details(id): # - /movies/:id에서 영화 정보를 가져옵니다.
    response = requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}")
    return response.json()
    
def get_movie_credits(id): # - /movies/:id/credits에서 출연진 및 제작진을 가져옵니다.
    
    response = requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits")
    return response.json()

def get_similar_movies(id): # - /movies/:id/similar 에서 유사한 영화를 조회합니다. 
    response = requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/similar")
    return response.json()

def find_movie_by_title(title: str):
    """제목(또는 overview)에 title 이 포함된 첫 번째 영화를 반환."""
    movies = get_popular_movies()
    query = title.lower()

    candidates = [
        m for m in movies
        if query in m.get("title", "").lower()
        or query in m.get("original_title", "").lower()
        or query in m.get("overview", "").lower()
    ]

    if not candidates:
        return None

    # 가장 인기(popularity) 높은 영화 하나 선택 (선택 기준은 취향대로 조정 가능)
    candidates.sort(key=lambda m: m.get("popularity", 0), reverse=True)
    return candidates[0]
    
def get_full_movie_info_by_title(title: str):
    """제목으로 찾아서: 기본정보 + 상세 + 출연/제작 + 유사 영화까지 묶어서 반환."""
    movie = find_movie_by_title(title)
    if movie is None:
        return {
            "error": f"'{title}'와(과) 일치하는 영화를 찾을 수 없습니다."
        }

    movie_id = movie["id"]

    details = get_movie_details(movie_id)
    credits = get_movie_credits(movie_id)
    similar = get_similar_movies(movie_id)

    return {
        "selected_movie": movie,   # /movies 에서 고른 기본 정보
        "details": details,        # /movies/{id}
        "credits": credits,        # /movies/{id}/credits
        "similar": similar,        # /movies/{id}/similar
    }

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits, 
    "get_similar_movies": get_similar_movies, 
    "get_full_movie_info_by_title": get_full_movie_info_by_title,
}

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get the list of popular movies.",
            "parameters": {
                "type": "object",
                "properties":{},
            }
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get the movie details with the specific id of a popular movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The id of the movie."
                    }
                }
            },
            "required": ["id"]
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "A function to get the credits for a movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The id of the movie."
                    }
                }
            },
            "required": ["id"]
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "A function to get similar movies for a movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The id of the movie."
                    }
                }
            },
            "required": ["id"]
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_full_movie_info_by_title",
            "description": "Given a movie title (Korean or English), "
                           "find the best-matching movie and return its details, "
                           "credits, and similar movies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the movie the user is asking about."
                    }
                },
                "required": ["title"],
            },
        },
    },       
]

In [4]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)   

In [5]:
while True:
    message = input("Send a message to the LLM: ")
    if message == "quit" or message == "q" :
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()
    

User: 영화 듄에 대한 정보를 알려줘
Calling function: get_full_movie_info_by_title with {"title":"듄"}
Ran get_full_movie_info_by_title with args {'title': '듄'} for a result of {'error': "'듄'와(과) 일치하는 영화를 찾을 수 없습니다."}
Calling function: get_full_movie_info_by_title with {"title":"Dune"}
Ran get_full_movie_info_by_title with args {'title': 'Dune'} for a result of {'error': "'Dune'와(과) 일치하는 영화를 찾을 수 없습니다."}
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/6YjnTRBz704LF1uJ3ZC4wsS9T8r.jpg', 'genre_ids': [28, 80, 53], 'id': 1290821, 'original_language': 'en', 'original_title': 'Shelter', 'overview': 'A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.', 'popularity': 506.2271, 'poster_path': 'https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZE

In [6]:
messages

[{'role': 'user', 'content': '영화 듄에 대한 정보를 알려줘'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_Cab4HR3TPTorjxYgLQ100UD3',
    'type': 'function',
    'function': {'name': 'get_full_movie_info_by_title',
     'arguments': '{"title":"듄"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_Cab4HR3TPTorjxYgLQ100UD3',
  'name': 'get_full_movie_info_by_title',
  'content': '{"error": "\'\\ub4c4\'\\uc640(\\uacfc) \\uc77c\\uce58\\ud558\\ub294 \\uc601\\ud654\\ub97c \\ucc3e\\uc744 \\uc218 \\uc5c6\\uc2b5\\ub2c8\\ub2e4."}'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_7e4of3x82ZV8cp3Wh4GiYF8i',
    'type': 'function',
    'function': {'name': 'get_full_movie_info_by_title',
     'arguments': '{"title":"Dune"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_7e4of3x82ZV8cp3Wh4GiYF8i',
  'name': 'get_full_movie_info_by_title',
  'content': '{"error": "\'Dune\'\\uc640(\\uacfc) \\uc77c\\uce58\\ud558\\ub294 \\uc601\\ud654\\ub97c \\ucc3e\\uc744 \\uc218 \\uc5